# Yelp Sentiment — Kaggle Runner

Comparative study: LSTM, Bi-LSTM, BERT, RoBERTa on 5-class Yelp Review Full.

**Before running:** open the right-hand panel and set
1. **Accelerator → GPU** (T4 x2 or P100)
2. **Internet → On**  (needed to clone the repo and download the dataset)

Then edit `REPO_URL` in the next cell and **Run All**.
All four models load the *same* persisted split, so the comparison is fair.

In [ ]:
# --- 1. Get the code -------------------------------------------------------
import os
REPO_URL = "https://github.com/mikiproda/yelp-sentiment-nn.git"
REPO_DIR = "/kaggle/working/NM"

if not os.path.exists(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull -q
%cd $REPO_DIR

# Kaggle already ships torch+CUDA, transformers, datasets, sklearn, seaborn.
# Install nothing heavy; just make sure datasets is recent enough.
!pip -q install -U "datasets>=2.19" >/dev/null 2>&1
import sys; sys.path.insert(0, "src")
print("repo ready at", REPO_DIR)

In [ ]:
# --- 2. Sanity: GPU available? --------------------------------------------
import torch
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# --- 3. Build the shared split (rebuilds the identical M1 split) ------------
# Loads Yelp from HuggingFace and writes data/splits/*.parquet + meta.json.
!python scripts/prepare_data.py

In [ ]:
# --- 4. LSTM ---------------------------------------------------------------
!python scripts/train_model.py --model lstm

In [ ]:
# --- 5. Bi-LSTM ------------------------------------------------------------
!python scripts/train_model.py --model bilstm

### Transformers
BERT and RoBERTa fine-tune via the Hugging Face `Trainer`. These cells run the
canonical seed (42). GPU strongly recommended.

In [ ]:
# --- 6. BERT (M4) ----------------------------------------------------------
!python scripts/train_model.py --model bert

In [ ]:
# --- 7. RoBERTa (M5) -------------------------------------------------------
!python scripts/train_model.py --model roberta

In [ ]:
# --- 7b. Robustness: extra training seeds (all four models) ----------------
# Cells 4-7 produced the canonical seed (42). Repeat all four models across
# additional seeds so the report can show mean +/- std. The DATA SPLIT is fixed
# (DataConfig.seed=42); only weight init + shuffling vary, so seeds stay fair.
#
# Keep [123, 7] for a solid n=3, or trim to [123] for a quicker n=2.
EXTRA_SEEDS = [123, 7]

for seed in EXTRA_SEEDS:
    for m in ["lstm", "bilstm", "bert", "roberta"]:
        print(f"\n########## {m}  seed={seed} ##########", flush=True)
        !python scripts/train_model.py --model {m} --seed {seed}

In [ ]:
# --- 7c. Aggregate all seeds into the comparison table ---------------------
!python scripts/build_report.py

In [ ]:
# --- 8. Package results for download --------------------------------------
# Zips results/ (metrics JSON + figures) into /kaggle/working for the Output tab.
import shutil
shutil.make_archive("/kaggle/working/results", "zip", "results")
print("Wrote /kaggle/working/results.zip — download it from the Output tab,")
print("or use 'Save Version' to persist the whole notebook output.")